# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](https://github.com/MartaLopezRoldan/lab-neural-networks/blob/master/your-code/tttboard.jpg?raw=1)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [41]:
import pandas as pd
import numpy as np

path = "https://raw.githubusercontent.com/MartaLopezRoldan/lab-neural-networks/master/your-code/tic-tac-toe.csv"
df = pd.read_csv(path)

In [13]:
df.head(25)

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True
5,x,x,x,x,o,o,b,b,o,True
6,x,x,x,x,o,b,o,o,b,True
7,x,x,x,x,o,b,o,b,o,True
8,x,x,x,x,o,b,b,o,o,True
9,x,x,x,x,b,o,o,o,b,True


In [16]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df)
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(df.columns))

df_encoded.head()

,TL_b,TL_o,TL_x,TM_b,TM_o,TM_x,TR_b,TR_o,TR_x,ML_b,...,BL_o,BL_x,BM_b,BM_o,BM_x,BR_b,BR_o,BR_x,class_False,class_True
0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
3,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [19]:
X_encoded = pd.DataFrame(df_encoded.drop(['class_False', 'class_True'], axis=1))
y = df['class'].astype(int)

In [24]:
X_scaled = (X_encoded - X_encoded.mean()) / X_encoded.std()
X_scaled.head()

,TL_b,TL_o,TL_x,TM_b,TM_o,TM_x,TR_b,TR_o,TR_x,ML_b,...,MR_x,BL_b,BL_o,BL_x,BM_b,BM_o,BM_x,BR_b,BR_o,BR_x
0,-0.521498,-0.732912,1.136009,-0.593918,-0.72452,1.238059,-0.521498,-0.732912,1.136009,-0.593918,...,-0.806873,-0.521498,-0.732912,1.136009,-0.593918,1.378783,-0.806873,-0.521498,1.362997,-0.879355
1,-0.521498,-0.732912,1.136009,-0.593918,-0.72452,1.238059,-0.521498,-0.732912,1.136009,-0.593918,...,-0.806873,-0.521498,1.362997,-0.879355,-0.593918,-0.724520,1.238059,-0.521498,1.362997,-0.879355
2,-0.521498,-0.732912,1.136009,-0.593918,-0.72452,1.238059,-0.521498,-0.732912,1.136009,-0.593918,...,-0.806873,-0.521498,1.362997,-0.879355,-0.593918,1.378783,-0.806873,-0.521498,-0.732912,1.136009
3,-0.521498,-0.732912,1.136009,-0.593918,-0.72452,1.238059,-0.521498,-0.732912,1.136009,-0.593918,...,-0.806873,-0.521498,1.362997,-0.879355,1.681976,-0.724520,-0.806873,1.915551,-0.732912,-0.879355
4,-0.521498,-0.732912,1.136009,-0.593918,-0.72452,1.238059,-0.521498,-0.732912,1.136009,-0.593918,...,-0.806873,1.915551,-0.732912,-0.879355,-0.593918,1.378783,-0.806873,1.915551,-0.732912,-0.879355


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [25]:
from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split


In [26]:
n_cols = X_scaled.shape[1]

In [33]:
def df_model():
    model = Sequential()
    model.add(Dense(50, activation='relu', input_shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(2))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [34]:
model=df_model()
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train, epochs=100, verbose=2)


Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


24/24 - 1s - 53ms/step - accuracy: 0.6279 - loss: 1.2355
Epoch 2/100
24/24 - 0s - 4ms/step - accuracy: 0.6958 - loss: 0.6627
Epoch 3/100
24/24 - 0s - 4ms/step - accuracy: 0.7441 - loss: 0.6041
Epoch 4/100
24/24 - 0s - 4ms/step - accuracy: 0.7572 - loss: 0.5673
Epoch 5/100
24/24 - 0s - 4ms/step - accuracy: 0.7676 - loss: 0.5131
Epoch 6/100
24/24 - 0s - 4ms/step - accuracy: 0.8029 - loss: 0.4252
Epoch 7/100
24/24 - 0s - 4ms/step - accuracy: 0.8251 - loss: 0.4157
Epoch 8/100
24/24 - 0s - 4ms/step - accuracy: 0.8316 - loss: 0.3688
Epoch 9/100
24/24 - 0s - 4ms/step - accuracy: 0.8551 - loss: 0.3333
Epoch 10/100
24/24 - 0s - 4ms/step - accuracy: 0.8695 - loss: 0.3209
Epoch 11/100
24/24 - 0s - 4ms/step - accuracy: 0.8420 - loss: 0.4412
Epoch 12/100
24/24 - 0s - 3ms/step - accuracy: 0.8890 - loss: 0.2742
Epoch 13/100
24/24 - 0s - 4ms/step - accuracy: 0.9021 - loss: 0.2606
Epoch 14/100
24/24 - 0s - 4ms/step - accuracy: 0.9099 - loss: 0.3334
Epoch 15/100
24/24 - 0s - 4ms/step - accuracy: 0.9373 

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [42]:
y_pred = model.predict(X_test)
y_pred_bool = np.argmax(y_pred, axis=1)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


In [43]:
from sklearn.metrics import accuracy_score, r2_score

accuracy = accuracy_score(y_test, y_pred_bool)

print('Accuracy:', round(accuracy, 4))

Accuracy: 0.724


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [ ]:
# your code here

**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
# your answer here